In [ ]:
import numpy as np
import pandas as pd
import os
import ms_entropy as me

In [2]:
df = pd.read_csv('/Users/ellayoung/Desktop/metabolo_confi_score/parsed_spectra.csv')

/var/folders/xd/3y34jslx1fsf1k1pd0bz8w3r0000gn/T/ipykernel_9142/446800534.py:1: DtypeWarning: Columns (11,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/ellayoung/Desktop/metabolo_confi_score/parsed_spectra.csv')


In [5]:
def parse_peaks_string(peaks_str):
    """
    Parse a peaks string like:
    '[[136.0435   3.4   ]\n [214.9619  38.4   ]\n ...]'
    into a list of lists of floats.
    """
    # Remove the outer brackets and any leading/trailing whitespace
    cleaned = peaks_str.strip("[] \n")
    # Split into lines (each line corresponds to one peak row)
    rows = cleaned.split('\n')
    result = []
    for row in rows:
        # Remove any remaining brackets and extra whitespace
        row_clean = row.strip(" []")
        if row_clean:
            # Split on whitespace and convert each element to float
            values = row_clean.split()
            result.append([float(val) for val in values])
    return result

def safe_parse_peaks(peaks):
    """
    Safely parse peaks, returning an empty list for invalid cases.
    """
    if isinstance(peaks, str):
        if "..." in peaks or peaks.strip() == "":
            return []  # Return empty list for invalid entries
        try:
            return parse_peaks_string(peaks)
        except ValueError:
            return []  # Handle unexpected format issues
    return peaks  # Already parsed peaks



In [6]:
df_clean = df.iloc[:, [0, 1, 10, 18, 13, 14, 16, 24]]

In [7]:
# only include entries with M+H or M-H adducts

df_clean_mh = df_clean[df_clean["Precursor Type"].isin(["[M+H]+", "[M-H]-"])]

In [8]:
df_clean_mh.info()

<class 'pandas.core.frame.DataFrame'>
Index: 773947 entries, 0 to 1934649
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Name            773947 non-null  object 
 1   peaks           773947 non-null  object 
 2   Precursor Type  773947 non-null  object 
 3   Exact Mass      773333 non-null  float64
 4   InChIKey        760653 non-null  object 
 5   SMILES          757768 non-null  object 
 6   Formula         773947 non-null  object 
 7   Spectrum ID     773947 non-null  int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 53.1+ MB


In [10]:
print(df_clean_mh.head())

                        Name  \
0  2-Guanidinopropionic acid   
1  2-Guanidinopropionic acid   
2  2-Guanidinopropionic acid   
3  2-Guanidinopropionic acid   
4  2-Guanidinopropionic acid   

                                               peaks Precursor Type  \
0  [[ 53.0135   2.4   ]\n [ 55.018   55.2   ]\n [...         [M+H]+   
1  [[ 53.0135   5.6   ]\n [ 55.018   82.6   ]\n [...         [M+H]+   
2  [[ 51.0231   8.7   ]\n [ 53.0136  12.1   ]\n [...         [M+H]+   
3  [[ 51.0231  23.8   ]\n [ 53.0136  30.9   ]\n [...         [M+H]+   
4  [[ 51.0231  76.4   ]\n [ 53.0136 110.    ]\n [...         [M+H]+   

   Exact Mass        InChIKey             SMILES   Formula  Spectrum ID  
0  131.069476  DVNFLGLGNLXITH  CC(NC(=N)N)C(=O)O  C4H9N3O2      3461655  
1  131.069476  DVNFLGLGNLXITH  CC(NC(=N)N)C(=O)O  C4H9N3O2      3461656  
2  131.069476  DVNFLGLGNLXITH  CC(NC(=N)N)C(=O)O  C4H9N3O2      3461657  
3  131.069476  DVNFLGLGNLXITH  CC(NC(=N)N)C(=O)O  C4H9N3O2      3461658  
4  131.069

In [30]:
import numpy as np

def normalize_spectrum(spectrum):
    """
    Normalize the intensities of the spectrum so the sum equals 1.
    Spectrum is assumed to be a NumPy array with shape (n, 2)
    where each row is (mz, intensity).
    """
    total_intensity = np.sum(spectrum[:, 1])
    if total_intensity > 0:
        spectrum[:, 1] /= total_intensity
    return spectrum

def find_best_match(exp_peak, library_spectrum, tolerance):
    """
    Find the best matching library peak to the exp_peak (within a tolerance),
    measured as the smallest absolute difference in m/z.
    
    Parameters:
        exp_peak: a tuple (mz, intensity)
        library_spectrum: NumPy array of shape (m, 2)
        tolerance: maximum allowed difference (in Da or ppm-based conversion) 
                   for a match.
                   
    Returns:
        matched_peak: tuple (mz, intensity) of the best match, or None if no match.
        index: index of the matched peak in the library_spectrum, or None.
    """
    exp_mz = exp_peak[0]
    differences = np.abs(library_spectrum[:, 0] - exp_mz)
    within_tol = differences <= tolerance
    if np.any(within_tol):
        idx = np.argmin(differences * within_tol.astype(float) + (~within_tol) * 1e6)
        return library_spectrum[idx], idx
    else:
        return None, None

def intensity_similarity(i1, i2):
    """
    Simple similarity factor between two intensities.
    Returns a number between 0 and 1, equal to 1 if both intensities are identical.
    """
    return min(i1, i2) / max(i1, i2) if max(i1, i2) > 0 else 0

def peak_by_peak_score(exp_spectrum, lib_spectrum, tolerance=0.01, alpha=0.5, beta=0.5, missing_penalty=0.1):
    """
    Compute a peak-by-peak score between an experimental spectrum and a library spectrum.
    
    Parameters:
      - exp_spectrum: NumPy array of shape (n,2): [mz, intensity]
      - lib_spectrum: NumPy array of shape (m,2): [mz, intensity]
      - tolerance: maximum allowed m/z difference for a match.
      - alpha: exponent applied to m/z values (to weight higher m/z more, for example).
      - beta: exponent applied to (normalized) intensity values.
      - missing_penalty: penalty factor subtracted when an experimental peak is not matched.
      
    Returns:
      - score: A numerical score (the higher, the better the match).
    """
    # Normalize both spectra so that intensities sum to 1.
    exp_spectrum = normalize_spectrum(exp_spectrum.copy())
    lib_spectrum = normalize_spectrum(lib_spectrum.copy())
    
    score = 0.0
    # Keep track of matched library peaks so they are not reused.
    lib_matched = np.zeros(len(lib_spectrum), dtype=bool)

    # Process each experimental peak.
    for exp_peak in exp_spectrum:
        matched_peak, idx = find_best_match(exp_peak, lib_spectrum, tolerance)
        weight = (exp_peak[0] ** alpha) * (exp_peak[1] ** beta)
        if matched_peak is not None and not lib_matched[idx]:
            # Calculate a similarity factor for the intensities.
            sim_factor = intensity_similarity(exp_peak[1], matched_peak[1])
            score += weight * sim_factor
            lib_matched[idx] = True  # Mark as used.
        else:
            # No matching library peak was found: subtract penalty.
            score -= missing_penalty * weight

    return score

# # Example usage:
# if __name__ == '__main__':
#     # Generate simple example spectra:
#     # Each row is [mz, intensity]. 
#     # For simplicity, we use small arrays; in practice these would have more peaks.
#     experimental = np.array([
#         [100.0, 50],
#         [200.0, 100],
#         [300.0, 30]
#     ], dtype=np.float32)
    
#     # Library spectrum: first two peaks match well; third peak is shifted.
#     library = np.array([
#         [100.0, 45],
#         [200.1, 95],
#         [300.5, 28]
#     ], dtype=np.float32)
    
#     # Set a tolerance of 0.2 (same mass unit as m/z values here).
#     score = peak_by_peak_score(experimental, library, tolerance=0.2, alpha=1.0, beta=1.0, missing_penalty=0.5)
#     print("Peak-by-Peak Score:", score)


In [ ]:
# def search_spectra_against_each_other(df_clean, ms1_tolerance_ppm=10, ms2_tolerance_da=0.05):
#     df_clean = me.compute_spectral_entropy(df_clean).copy()
#     df_clean["Exact Mass"] = df_clean["Exact Mass"].astype(float)
#     df_clean = df_clean.sort_values(by="Exact Mass").reset_index(drop=True)
#     precursor_mz_array = df_clean["Exact Mass"].values
#     all_results = []

#     for idx, row in df_clean.iterrows():
#         precursor_mz = row["Exact Mass"]
#         peaks = row["peaks"]
#         inchikey = row["InChIKey"]
#         spec_id = row["Spectrum ID"]
#         entropy_bin = row["entropy_bin"]

#         precursor_mz_left = precursor_mz * (1 - ms1_tolerance_ppm / 1e6)
#         precursor_mz_right = precursor_mz * (1 + ms1_tolerance_ppm / 1e6)

#         candidate_idx_left = np.searchsorted(precursor_mz_array, precursor_mz_left, side="left")
#         candidate_idx_right = np.searchsorted(precursor_mz_array, precursor_mz_right, side="right")

#         exp_spectrum = np.array(peaks)

#         for candidate_idx in range(candidate_idx_left, candidate_idx_right):
#             if candidate_idx == idx:
#                 continue

#             candidate_row = df_clean.iloc[candidate_idx]
#             candidate_peaks = candidate_row["peaks"]
#             lib_spectrum = np.array(candidate_peaks)

#             # Existing entropy similarity calculation
#             entropy_similarity = ms_entropy.calculate_entropy_similarity(
#                 peaks, candidate_peaks, ms2_tolerance_in_da=ms2_tolerance_da
#             )

#             # Peak-by-peak similarity calculation (new method)
#             pbp_score = peak_by_peak_score(
#                 exp_spectrum, lib_spectrum, tolerance=ms2_tolerance_da
#             )

#             all_results.append([
#                 spec_id, inchikey, candidate_row["Spectrum ID"], candidate_row["InChIKey"],
#                 entropy_similarity, pbp_score, entropy_bin
#             ])

#     df_results = pd.DataFrame(all_results, columns=[
#         "spec_id_query", "inchikey_query", "spec_id_library", "inchikey_library",
#         "score_entropy", "score_peak_by_peak", "entropy_bin"
#     ])

#     # Assign correctness label directly
#     df_results["correct"] = df_results["inchikey_query"] == df_results["inchikey_library"]

#     # For each query-library pair, keep the highest entropy score
#     df_results = df_results.sort_values(by="score_entropy", ascending=False)
#     df_results = df_results.groupby(["spec_id_query", "inchikey_library"]).first().reset_index()

#     return df_results


In [ ]:
df_short = df_clean_mh.sample(n=100000, random_state=42)

In [32]:
import numpy as np
import pandas as pd
import ms_entropy
from tqdm import tqdm

# Parse peaks safely using provided functions
df_short['parsed_peaks'] = df_short['peaks'].apply(safe_parse_peaks)

# Keep only valid spectra
df_short = df_short[df_short['parsed_peaks'].apply(lambda x: len(x) > 0 and isinstance(x, list))]
df_short = df_short.dropna(subset=['Exact Mass']).reset_index(drop=True)

# Sorting for efficient searching
df_sorted = df_short.sort_values(by='Exact Mass').reset_index(drop=True)
mz_array = df_sorted['Exact Mass'].values

# Similarity calculation functions (provided earlier)
def normalize_spectrum(spectrum):
    spectrum = np.array(spectrum, dtype=float)
    total_intensity = np.sum(spectrum[:, 1])
    if total_intensity > 0:
        spectrum[:, 1] /= total_intensity
    return spectrum

def find_best_match(exp_peak, library_spectrum, tolerance):
    differences = np.abs(library_spectrum[:, 0] - exp_peak[0])
    within_tol = differences <= tolerance
    if np.any(within_tol):
        idx = np.argmin(differences + (~within_tol) * 1e6)
        return library_spectrum[idx], idx
    else:
        return None, None

def intensity_similarity(i1, i2):
    return min(i1, i2) / max(i1, i2) if max(i1, i2) > 0 else 0

def peak_by_peak_score(exp_spectrum, lib_spectrum, tolerance=0.01, alpha=0.5, beta=0.5, missing_penalty=0.1):
    exp_spectrum = normalize_spectrum(exp_spectrum)
    lib_spectrum = normalize_spectrum(lib_spectrum)
    score = 0.0
    lib_matched = np.zeros(len(lib_spectrum), dtype=bool)

    for exp_peak in exp_spectrum:
        matched_peak, idx = find_best_match(exp_peak, lib_spectrum, tolerance)
        weight = (exp_peak[0] ** alpha) * (exp_peak[1] ** beta)
        if matched_peak is not None and not lib_matched[idx]:
            sim_factor = intensity_similarity(exp_peak[1], matched_peak[1])
            score += weight * sim_factor
            lib_matched[idx] = True
        else:
            score -= missing_penalty * weight
    return score


In [33]:

# Main similarity calculation loop
results = []

ppm_tolerance = 10
ms2_tolerance_da = 0.05

for idx, query_row in tqdm(df_sorted.iterrows(), total=len(df_sorted)):
    query_mz = query_row['Exact Mass']
    query_peaks = query_row['parsed_peaks']
    query_id = query_row['Spectrum ID']
    query_inchikey = query_row['InChIKey']

    mz_lower = query_mz * (1 - ppm_tolerance / 1e6)
    mz_upper = query_mz * (1 + ppm_tolerance / 1e6)

    left = np.searchsorted(mz_array, mz_lower, side="left")
    right = np.searchsorted(mz_array, mz_upper, side="right")

    query_spectrum_np = np.array(query_peaks)

    for candidate_idx in range(left, right):
        if candidate_idx == idx:
            continue  # Skip self-comparison

        candidate_row = df_sorted.iloc[candidate_idx]
        candidate_peaks = candidate_row['parsed_peaks']
        candidate_spectrum_np = np.array(candidate_peaks)
        candidate_id = candidate_row['Spectrum ID']
        candidate_inchikey = candidate_row['InChIKey']

        # Entropy Similarity
        entropy_score = ms_entropy.calculate_entropy_similarity(
            query_peaks, candidate_peaks, ms2_tolerance_in_da=ms2_tolerance_da
        )

        # Peak-by-Peak Similarity
        pbp_score = peak_by_peak_score(
            query_spectrum_np, candidate_spectrum_np, tolerance=ms2_tolerance_da
        )

        # Append scores
        results.append({
            'spec_id_query': query_id,
            'inchikey_query': query_inchikey,
            'spec_id_candidate': candidate_id,
            'inchikey_candidate': candidate_inchikey,
            'entropy_similarity': entropy_score,
            'peak_by_peak_score': pbp_score,
            'correct_match': query_inchikey == candidate_inchikey
        })

# Convert results to DataFrame
similarity_df = pd.DataFrame(results)

# Optional: Keep best matches per query-candidate pair
similarity_df = similarity_df.sort_values(by='entropy_similarity', ascending=False).groupby(
    ['spec_id_query', 'spec_id_candidate'], as_index=False).first()

print(similarity_df.head())


100%|██████████| 99873/99873 [24:27<00:00, 68.05it/s]  


   spec_id_query  spec_id_candidate  inchikey_query inchikey_candidate  \
0        1035217            1035219  SNKAWJBJQDLSFF     SNKAWJBJQDLSFF   
1        1035217            1481649  SNKAWJBJQDLSFF     SNKAWJBJQDLSFF   
2        1035219            1035217  SNKAWJBJQDLSFF     SNKAWJBJQDLSFF   
3        1035219            1481649  SNKAWJBJQDLSFF     SNKAWJBJQDLSFF   
4        1035264            1035267  NVWYLRLKCCCUBZ     NVWYLRLKCCCUBZ   

   entropy_similarity  peak_by_peak_score  correct_match  
0            0.953537           24.071697           True  
1            0.758275            9.441221           True  
2            0.953537           24.242164           True  
3            0.693516           12.264676           True  
4            0.871399           16.992299           True  


In [18]:
similarity_df.describe()

,spec_id_query,spec_id_candidate,entropy_similarity,peak_by_peak_score
count,1.593800e+04,1.593800e+04,15938.000000,15938.000000
mean,2.855363e+06,2.855363e+06,0.201527,-39.710495
std,1.241492e+06,1.241492e+06,0.259778,68.477277
min,1.035264e+06,1.035264e+06,0.000000,-573.838363
25%,1.579444e+06,1.579444e+06,0.000000,-75.890224
50%,3.189576e+06,3.189576e+06,0.085878,-44.515537
75%,4.016228e+06,4.016228e+06,0.316932,-7.174320
max,4.656499e+06,4.656499e+06,0.999997,438.712191


In [34]:
y_true = similarity_df['correct_match'].astype(int)  # convert True/False to 1/0
entropy_scores = similarity_df['entropy_similarity']
pbp_scores = similarity_df['peak_by_peak_score']


In [35]:
from sklearn.metrics import roc_auc_score

auc_entropy = roc_auc_score(y_true, entropy_scores)
auc_pbp = roc_auc_score(y_true, pbp_scores)

print(f"Entropy similarity ROC-AUC: {auc_entropy:.4f}")
print(f"Peak-by-peak ROC-AUC: {auc_pbp:.4f}")


Entropy similarity ROC-AUC: 0.6923
Peak-by-peak ROC-AUC: 0.6659


In [36]:
from sklearn.metrics import average_precision_score

ap_entropy = average_precision_score(y_true, entropy_scores)
ap_pbp = average_precision_score(y_true, pbp_scores)

print(f"Entropy similarity Average Precision: {ap_entropy:.4f}")
print(f"Peak-by-peak Average Precision: {ap_pbp:.4f}")


Entropy similarity Average Precision: 0.3896
Peak-by-peak Average Precision: 0.3484


In [37]:
corr = similarity_df[['entropy_similarity', 'peak_by_peak_score']].corr().iloc[0,1]
print(f"Correlation between the two scores: {corr:.4f}")


Correlation between the two scores: 0.7575


In [24]:
def scoring_function(y_true, scores):
    return roc_auc_score(y_true, scores)
import itertools
from sklearn.metrics import roc_auc_score

# Possible values to tune
param_grid = {
    'tolerance': [0.01, 0.02, 0.05, 0.1],
    'alpha': [0.5, 1.0, 1.5],
    'beta': [0.5, 1.0, 2.0],
    'missing_penalty': [0.1, 0.5, 1.0, 2.0],
}

param_combinations = list(itertools.product(
    param_grid['tolerance'],
    param_grid['alpha'],
    param_grid['beta'],
    param_grid['missing_penalty']
))


In [29]:
# Step 0: Build dictionary once
spectrum_dict = dict(zip(df_sorted['Spectrum ID'], df_sorted['parsed_peaks']))

# Step 1: Tuning loop
best_score = -np.inf
best_params = None

for tolerance, alpha, beta, missing_penalty in tqdm(param_combinations):
    # Step 2: Define compute_score inside loop but after spectrum_dict is defined
    def compute_score(row):
        qid = row['spec_id_query']
        cid = row['spec_id_candidate']
        if qid not in spectrum_dict or cid not in spectrum_dict:
            return np.nan
        return peak_by_peak_score(
            np.array(spectrum_dict[qid]),
            np.array(spectrum_dict[cid]),
            tolerance=tolerance,
            alpha=alpha,
            beta=beta,
            missing_penalty=missing_penalty
        )

    # Step 3: Apply scoring
    similarity_df['tuned_pbp_score'] = similarity_df.apply(compute_score, axis=1)

    # Step 4: Evaluate performance
    mask = similarity_df['tuned_pbp_score'].notna()
    auc = roc_auc_score(similarity_df.loc[mask, 'correct_match'], similarity_df.loc[mask, 'tuned_pbp_score'])

    if auc > best_score:
        best_score = auc
        best_params = {
            'tolerance': tolerance,
            'alpha': alpha,
            'beta': beta,
            'missing_penalty': missing_penalty
        }

print("✅ Best ROC-AUC:", best_score)
print("🏆 Best parameters:", best_params)



  0%|          | 0/144 [00:00<?, ?it/s]

100%|██████████| 144/144 [25:24<00:00, 10.58s/it]

✅ Best ROC-AUC: 0.6854786878487505
🏆 Best parameters: {'tolerance': 0.01, 'alpha': 0.5, 'beta': 0.5, 'missing_penalty': 0.1}
